In [0]:
%run ./ingest_common_helpers

In [0]:
%run ./ingest_manifest_writer

In [0]:
# Arbeitnow API Configuration
ARBEITNOW_API = "https://www.arbeitnow.com/api/job-board-api"
SOURCE_NAME = "arbeitnow"

# Arbeitnow-specific field mapping
ARBEITNOW_FIELDS = {
    'job_id': 'slug',           # string - unique job identifier
    'company': 'company_name',   # string
    'title': 'title',            # string
    'description': 'description', # string
    'remote': 'remote',          # bool
    'url': 'url',                # string
    'tags': 'tags',              # array/object
    'job_types': 'job_types',    # array/object
    'location': 'location',      # string
    'created_at': 'created_at'   # int64 - Unix timestamp
}

In [0]:
def extract_arbeitnow_job_id(job):
    """Extract job ID from Arbeitnow record"""
    return job.get('slug', '')

def validate_arbeitnow_record(job):
    """Validate Arbeitnow job record structure"""
    required_fields = ['slug', 'title', 'company_name']
    for field in required_fields:
        if not job.get(field):
            return False, f"Missing required field: {field}"
    
    # Validate data types
    if not isinstance(job.get('created_at'), int):
        return False, "created_at must be int64 (Unix timestamp)"
    
    if not isinstance(job.get('remote'), bool):
        return False, "remote must be boolean"
    
    return True, None

def fetch_arbeitnow_jobs():
    """Fetch jobs from Arbeitnow API"""
    try:
        response = requests.get(ARBEITNOW_API, timeout=30)
        response.raise_for_status()
        data = response.json()
        jobs = data.get('data', [])
        
        # Validate records
        validated_jobs = []
        for job in jobs:
            is_valid, error = validate_arbeitnow_record(job)
            if is_valid:
                validated_jobs.append(job)
            else:
                print(f"  WARNING: Skipping invalid record - {error}")
        
        return validated_jobs, None
    except Exception as e:
        return None, str(e)

In [0]:
print("LMIP BRONZE LAYER INGESTION - ARBEITNOW")
print("Started:", datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'))
print("Target:", BRONZE_JOB_SNAPSHOT)


# Execute ingestion to Bronze layer
arbeitnow_success, batch_id, records_count = ingest_to_bronze(
    source_name=SOURCE_NAME,
    fetch_function=fetch_arbeitnow_jobs,
    api_url=ARBEITNOW_API,
    extract_job_id_func=extract_arbeitnow_job_id,
    log_api_response_func=log_api_response,
    start_pipeline_run_func=start_pipeline_run,
    complete_pipeline_run_func=complete_pipeline_run,
    log_audit_func=log_audit_pipeline_run
)

print(f"Status: {'SUCCESS' if arbeitnow_success else 'FAILED'}")
if arbeitnow_success:
    print(f"Batch ID: {batch_id}")
    print(f"Records: {records_count}")
print(f"Completed: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")


In [0]:
%sql
-- View recent Arbeitnow bronze job snapshots
SELECT 
  source_name,
  batch_id,
  COUNT(*) as record_count,
  MIN(ingestion_timestamp) as first_ingested,
  MAX(ingestion_timestamp) as last_ingested
FROM bronze.bronze_job_snapshot
WHERE source_name = 'arbeitnow'
GROUP BY source_name, batch_id
ORDER BY last_ingested DESC
LIMIT 10

In [0]:
%sql
-- View Arbeitnow pipeline audit history
SELECT 
  run_id as batch_id,
  pipeline_name,
  status,
  start_time,
  end_time,
  rows_read,
  rows_written,
  ROUND(runtime_seconds, 2) as runtime_sec,
  error_message
FROM audit.audit_pipeline_runs
WHERE pipeline_name = 'bronze_ingestion_arbeitnow'
ORDER BY start_time DESC
LIMIT 20

In [0]:
%sql
-- View Arbeitnow API response telemetry
SELECT 
  source_name,
  batch_id,
  http_status_code,
  response_time_ms,
  retry_count,
  rate_limit_hit,
  logged_at
FROM bronze.bronze_api_response_log
WHERE source_name = 'arbeitnow'
ORDER BY logged_at DESC
LIMIT 20